In [0]:
# ============================================================
# Bronze — Source 12: Reviews / Tickets
#
# Incremental load using watermark pattern.
#
# Source: s3://ecommerce-lakehouse-467091806172-raw-01/source=12_reviews_tickets/
# Target: bronze.reviews catalog in Unity Catalog
#
# Tables:
#   - bronze.reviews.reviews
#   - bronze.reviews.tickets
#
# Raw format: JSON (daily partitioned)
# ============================================================

import sys
sys.path.append("/Workspace/Users/sutharripal26@gmail.com/ecommerce-lakehouse/pipelines/bronze/shared")
from bronze_utils import get_watermark, update_watermark
from pyspark.sql.functions import col, lit, max as spark_max

spark.conf.set("spark.sql.legacy.parquet.nanosAsLong", "true")

RAW_BUCKET = "s3://ecommerce-lakehouse-467091806172-raw-01"
SOURCE = "12_reviews_tickets"

dbutils.fs.ls(f"{RAW_BUCKET}/source={SOURCE}/year=2026/month=04/day=18/")

In [0]:
# Cell 2 — Read reviews and tickets JSON files
watermark = get_watermark(spark, SOURCE)
print(f"Loading files modified after: {watermark}")

path_reviews = f"{RAW_BUCKET}/source={SOURCE}/year=2026/month=04/day=*/reviews_*.json"
path_tickets = f"{RAW_BUCKET}/source={SOURCE}/year=2026/month=04/day=*/tickets_*.json"

df_reviews = spark.read.format("json") \
    .load(path_reviews) \
    .filter(col("_metadata.file_modification_time") > lit(watermark))

df_tickets = spark.read.format("json") \
    .load(path_tickets) \
    .filter(col("_metadata.file_modification_time") > lit(watermark))

print(f"Reviews rows: {df_reviews.count()}")
print(f"Tickets rows: {df_tickets.count()}")
df_reviews.printSchema()

In [0]:

# Cell 3 — Flatten reviews and tickets with correct schemas
from pyspark.sql.functions import explode

# Flatten reviews
reviews_flat = df_reviews \
    .select(explode(col("records")).alias("r")) \
    .select(
        col("r.review_id").alias("record_id"),
        col("r.order_id"),
        col("r.product_sku"),
        col("r.customer_id"),
        col("r.rating"),
        col("r.title"),
        col("r.body"),
        col("r.helpful_votes"),
        col("r.verified_purchase"),
        col("r.created_at"),
        col("r.record_type"),
        col("r._source").alias("source"),
    )

# Check tickets schema first
df_tickets.select(explode(col("records")).alias("r")).select("r.*").printSchema()

In [0]:
# Cell 4 — Flatten tickets and write both to Bronze
tickets_flat = df_tickets \
    .select(explode(col("records")).alias("r")) \
    .select(
        col("r.ticket_id").alias("record_id"),
        col("r.order_id"),
        col("r.customer_id"),
        col("r.subject").alias("title"),
        col("r.body"),
        col("r.status"),
        col("r.priority"),
        col("r.category"),
        col("r.channel"),
        col("r.agent_id"),
        col("r.created_at"),
        col("r.resolved_at"),
        col("r.record_type"),
        col("r._source").alias("source"),
    )

print(f"Reviews: {reviews_flat.count()} rows")
print(f"Tickets: {tickets_flat.count()} rows")

# Write to Bronze
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze.reviews")

reviews_flat.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("bronze.reviews.reviews")

tickets_flat.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("bronze.reviews.tickets")

latest_ts = df_reviews.select(spark_max("_metadata.file_modification_time")).collect()[0][0]
total_rows = reviews_flat.count() + tickets_flat.count()
update_watermark(spark, SOURCE, latest_ts, total_rows)
print(f"✅ bronze.reviews.reviews: {reviews_flat.count()} rows")
print(f"✅ bronze.reviews.tickets: {tickets_flat.count()} rows")